# Colab Dense Depth Probes for Experiment 1

This notebook trains the new dense per-patch relative-depth probes on Google Colab from a zipped copy of the current repo stored in Google Drive.

It expects the repo zip to contain `configs/`, `scripts/`, `exp1/`, and the Experiment 1 data directory if you want to train immediately. Dense-depth training needs:

- `data/exp1_bounded/manifests/render_valid.parquet`
- `data/exp1_bounded/renders/.../{depth.npy,mask.npy}`
- patch-token feature caches like `data/exp1_bounded/features/dinov2_vit_b/final_patch.npz`

If patch caches are missing, the notebook can run `scripts/extract_exp1_patch_features.py`, but full-dataset patch extraction can be very large. Prefer using existing patch caches when available, especially for DINOv2 at 518px.

In [ ]:
from pathlib import Path

# Required: zipped current repo in Google Drive.
REPO_ZIP_IN_DRIVE = "/content/drive/MyDrive/cv-project.zip"

# Persistent output location in Drive. Training outputs are copied here.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/exp1_dense_depth_outputs"

# Local Colab workspace. This is fast but ephemeral.
WORK_DIR = Path("/content/exp1_dense_depth_colab")
REPO_EXTRACT_DIR = WORK_DIR / "repo_extract"
REPO_DIR = WORK_DIR / "cv-project"
LOCAL_OUTPUT_DIR = WORK_DIR / "outputs" / "dense_depth_probes"

# Repo config and data assumptions.
CONFIG_RELATIVE_PATH = "configs/exp1_bounded.yaml"
DATA_RELATIVE_PATH = "data/exp1_bounded"

# Dense-depth model/layer selection.
# These must have patch caches named <layer>_patch.npz, or patch extraction must run.
MODELS = ["dinov2_vit_b"]
DENSE_LAYERS = ["final", "layer8"]

# Training mode:
#   "smoke_one" trains one model/layer/texture for 1-2 epochs to verify everything.
#   "selected_texture" trains all selected models/layers for one texture condition.
#   "full_grid" trains the configured within-texture and cross-texture grid.
RUN_MODE = "smoke_one"
SMOKE_MODEL = MODELS[0]
SMOKE_LAYER = DENSE_LAYERS[0]
SMOKE_TEXTURE = "flat"

# Used for selected_texture mode. Ignored for full_grid.
TEXTURE_CONDITION = ["flat"]

# Dense-depth training settings. Set EPOCHS=None to use config defaults.
EPOCHS = 1 if RUN_MODE == "smoke_one" else None
TRAIN_BATCH_SIZE = 16  # Try 32 on larger GPUs; reduce if Colab runs out of memory.
DEVICE = "cuda"

# Patch extraction settings. Leave false if the repo zip already contains *_patch.npz caches.
EXTRACT_MISSING_PATCH_CACHES = False
PATCH_BATCH_SIZE = 4
PATCH_NUM_WORKERS = 4
PATCH_DTYPE = "float16"
PATCH_LIMIT = None  # For debugging only, e.g. 512. Full training needs full caches.

print("Configured local workspace:", WORK_DIR)

## Mount Drive and Check Runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is available. In Colab, switch Runtime type to GPU.")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=False)

## Install Dependencies

This installs the Python packages needed for patch feature extraction and dense probe training. It intentionally skips Blender/rendering dependencies.

In [ ]:
packages = [
    "transformers>=4.38.0",
    "huggingface_hub>=0.20.0",
    "open_clip_torch>=2.24.0",
    "hydra-core>=1.3.2",
    "omegaconf>=2.3.0",
    "pandas>=2.0.0",
    "pyarrow>=14.0.0",
    "Pillow>=10.0.0",
    "tqdm>=4.66.0",
    "scikit-learn>=1.3.0",
    "matplotlib>=3.8.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Dependencies installed.")

## Extract Repo Zip to Local Disk

In [ ]:
import json
import shutil
import zipfile

def run(cmd, *, cwd=None, env=None):
    printable = " ".join(str(x) for x in cmd)
    print("+", printable, flush=True)
    process = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {printable}")

def safe_extract_zip(zip_path: Path, dest: Path) -> None:
    zip_path = Path(zip_path)
    dest = Path(dest)
    if not zip_path.is_file():
        raise FileNotFoundError(f"Missing repo zip: {zip_path}")
    dest.mkdir(parents=True, exist_ok=True)
    resolved_dest = dest.resolve()
    print(f"Extracting {zip_path} -> {dest}")
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (dest / member.filename).resolve()
            if not str(target).startswith(str(resolved_dest)):
                raise RuntimeError(f"Unsafe zip member: {member.filename}")
        zf.extractall(dest)

def find_repo_root(root: Path) -> Path:
    hits = sorted(root.rglob("scripts/train_all_dense_depth_probes.py"))
    if not hits:
        raise FileNotFoundError("Could not find scripts/train_all_dense_depth_probes.py in repo zip")
    return hits[0].parents[1]

WORK_DIR.mkdir(parents=True, exist_ok=True)
if REPO_EXTRACT_DIR.exists():
    print("Removing previous extracted repo:", REPO_EXTRACT_DIR)
    shutil.rmtree(REPO_EXTRACT_DIR)
if REPO_DIR.exists():
    print("Removing previous repo dir:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

safe_extract_zip(Path(REPO_ZIP_IN_DRIVE), REPO_EXTRACT_DIR)
found_repo = find_repo_root(REPO_EXTRACT_DIR)
shutil.move(str(found_repo), str(REPO_DIR))

sys.path.insert(0, str(REPO_DIR))
os.environ["CV_PROJECT_ROOT"] = str(REPO_DIR)
os.environ.setdefault("HF_HOME", str(WORK_DIR / "hf_cache"))
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

CONFIG_PATH = REPO_DIR / CONFIG_RELATIVE_PATH
DATA_ROOT = REPO_DIR / DATA_RELATIVE_PATH
print("Repo ready:", REPO_DIR)
print("Config:", CONFIG_PATH)
print("Data root:", DATA_ROOT)
print("HF_HOME:", os.environ["HF_HOME"])

for script in ["scripts/extract_exp1_patch_features.py", "scripts/train_dense_depth_probe.py", "scripts/train_all_dense_depth_probes.py"]:
    path = REPO_DIR / script
    if not path.is_file():
        raise FileNotFoundError(f"Missing expected script: {path}")

## Rewrite Manifest Paths for Colab

If the repo zip was made on another machine, the manifest may contain absolute paths like `/Users/jerry/...`. This cell rewrites any path containing `data/exp1_bounded/` to the local Colab extraction path and saves a backup of the original manifest.

In [ ]:
import pandas as pd

manifest_path = DATA_ROOT / "manifests" / "render_valid.parquet"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Missing render_valid manifest: {manifest_path}")

backup_path = manifest_path.with_suffix(".original.parquet")
if not backup_path.is_file():
    shutil.copy2(manifest_path, backup_path)

df = pd.read_parquet(manifest_path)
marker = DATA_RELATIVE_PATH.rstrip("/") + "/"

def remap_exp1_path(value):
    if pd.isna(value):
        return value
    text = str(value)
    if marker in text:
        rel = text.split(marker, 1)[1]
        return str((DATA_ROOT / rel).resolve())
    return text

path_columns = [col for col in df.columns if col.endswith("_path")]
for col in path_columns:
    df[col] = df[col].map(remap_exp1_path)

required_columns = ["render_id", "split", "texture_condition", "rgb_path", "depth_path", "mask_path"]
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"Manifest is missing required columns: {missing_cols}")

for col in ["rgb_path", "depth_path", "mask_path"]:
    exists = df[col].head(50).map(lambda p: Path(str(p)).is_file())
    if not exists.all():
        bad = df.loc[~exists, ["render_id", col]].head(5)
        raise FileNotFoundError(f"Sample {col} paths are missing after rewrite:\n{bad}")

df.to_parquet(manifest_path, index=False)
print(f"Rewrote manifest paths in {manifest_path}")
print("Rows:", len(df), "Unique render_ids:", df["render_id"].nunique())
display(df.groupby(["split", "texture_condition"]).size().rename("rows").reset_index())

## Validate Patch Caches

Dense depth uses patch-token caches named `<layer>_patch.npz`. Global feature caches like `final.npz` are not enough for this task.

In [ ]:
import numpy as np

feature_root = DATA_ROOT / "features"
required_patch_caches = [
    feature_root / model / f"{layer}_patch.npz"
    for model in MODELS
    for layer in DENSE_LAYERS
]

def inspect_patch_cache(path: Path):
    if not path.is_file():
        return {"path": str(path), "exists": False}
    with np.load(path, allow_pickle=False, mmap_mode="r") as data:
        ids = data["render_ids"].astype(str)
        patches = data["patch_features"]
        return {
            "path": str(path),
            "exists": True,
            "shape": tuple(int(x) for x in patches.shape),
            "dtype": str(patches.dtype),
            "unique_ids": len(set(ids.tolist())),
            "ids": len(ids),
            "grid": tuple(int(x) for x in data["patch_grid_shape"].tolist()) if "patch_grid_shape" in data.files else tuple(patches.shape[1:3]),
            "feature_dim": int(np.asarray(data["feature_dim"]).item()) if "feature_dim" in data.files else int(patches.shape[-1]),
        }

cache_summary = [inspect_patch_cache(path) for path in required_patch_caches]
display(pd.DataFrame(cache_summary))

missing_patch_caches = [Path(row["path"]) for row in cache_summary if not row["exists"]]
if missing_patch_caches:
    print("Missing patch caches:")
    for path in missing_patch_caches:
        print("  ", path)
    if not EXTRACT_MISSING_PATCH_CACHES:
        print("Set EXTRACT_MISSING_PATCH_CACHES=True to extract them, or upload a repo zip containing them.")
else:
    print("All requested patch caches are present.")

## Optional: Extract Missing Patch Caches

This can require substantial RAM and disk. For the full bounded dataset, CLIP patch caches are much smaller than DINOv2 patch caches because DINOv2 defaults to 518px inputs.

In [ ]:
if missing_patch_caches and EXTRACT_MISSING_PATCH_CACHES:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "extract_exp1_patch_features.py"),
        "--config", str(CONFIG_PATH),
        "--render-manifest", str(manifest_path),
        "--feature-dir", str(feature_root),
        "--models", *MODELS,
        "--layers", *DENSE_LAYERS,
        "--device", DEVICE,
        "--batch-size", str(PATCH_BATCH_SIZE),
        "--num-workers", str(PATCH_NUM_WORKERS),
        "--dtype", PATCH_DTYPE,
        "--allow-unvalidated",
    ]
    if PATCH_LIMIT is not None:
        cmd.extend(["--limit", str(int(PATCH_LIMIT))])
    env = os.environ.copy()
    env["CV_PROJECT_ROOT"] = str(REPO_DIR)
    env["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    run(cmd, cwd=REPO_DIR, env=env)
elif missing_patch_caches:
    raise RuntimeError("Patch caches are missing and EXTRACT_MISSING_PATCH_CACHES=False")
else:
    print("Skipping patch extraction because all requested caches exist.")

## Smoke Validate Dense Dataset Joins

In [ ]:
from exp1.tasks.dense_relative_depth import Exp1DenseDepthDataset

manifest = pd.read_parquet(manifest_path)
rows = []
for model in MODELS:
    for layer in DENSE_LAYERS:
        patch_cache = feature_root / model / f"{layer}_patch.npz"
        if not patch_cache.is_file():
            rows.append({"model": model, "layer": layer, "split": "missing_cache", "rows": 0})
            continue
        for split in ["train", "val", "test"]:
            dataset = Exp1DenseDepthDataset(
                patch_cache,
                manifest=manifest,
                split=split,
                texture_condition=TEXTURE_CONDITION if RUN_MODE == "selected_texture" else None,
                project_root=REPO_DIR,
                cache_targets=False,
            )
            rows.append({"model": model, "layer": layer, "split": split, "rows": len(dataset), "grid": dataset.patch_grid_shape, "feature_dim": dataset.feature_dim})
display(pd.DataFrame(rows))

## Train Dense Depth Probes

In [ ]:
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["CV_PROJECT_ROOT"] = str(REPO_DIR)
env["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + env.get("PYTHONPATH", "")

def add_common_train_args(cmd):
    if EPOCHS is not None:
        cmd.extend(["--epochs", str(int(EPOCHS))])
    if TRAIN_BATCH_SIZE is not None:
        cmd.extend(["--batch-size", str(int(TRAIN_BATCH_SIZE))])
    cmd.extend(["--device", DEVICE])
    return cmd

if RUN_MODE == "smoke_one":
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "train_dense_depth_probe.py"),
        "--config", str(CONFIG_PATH),
        "--model", SMOKE_MODEL,
        "--layer", SMOKE_LAYER,
        "--render-manifest", str(manifest_path),
        "--output-dir", str(LOCAL_OUTPUT_DIR / "smoke"),
        "--texture-condition", SMOKE_TEXTURE,
    ]
    add_common_train_args(cmd)
elif RUN_MODE in {"selected_texture", "full_grid"}:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "train_all_dense_depth_probes.py"),
        "--config", str(CONFIG_PATH),
        "--models", *MODELS,
        "--layers", *DENSE_LAYERS,
        "--output-dir", str(LOCAL_OUTPUT_DIR),
        "--fail-on-missing",
    ]
    if RUN_MODE == "selected_texture":
        cmd.extend(["--texture-condition", *TEXTURE_CONDITION])
    add_common_train_args(cmd)
else:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

run(cmd, cwd=REPO_DIR, env=env)

## Copy Outputs Back to Drive

In [ ]:
drive_output = Path(DRIVE_OUTPUT_DIR)
drive_output.mkdir(parents=True, exist_ok=True)
target = drive_output / f"dense_depth_{RUN_MODE}"
if target.exists():
    shutil.rmtree(target)
shutil.copytree(LOCAL_OUTPUT_DIR, target)
print("Copied dense-depth outputs to:", target)

## Summarize Metrics

In [ ]:
metric_rows = []
for metrics_path in sorted(LOCAL_OUTPUT_DIR.rglob("metrics.json")):
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    metadata = payload.get("metadata", {})
    metrics = payload.get("metrics", {})
    for split, values in metrics.items():
        row = {
            "metrics_path": str(metrics_path.relative_to(LOCAL_OUTPUT_DIR)),
            "model": metadata.get("model_name"),
            "layer": metadata.get("layer_name"),
            "split": split,
            "texture_condition": metadata.get("texture_condition"),
            "train_texture_condition": metadata.get("train_texture_condition"),
            "eval_texture_condition": metadata.get("eval_texture_condition"),
            **values,
        }
        metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
if metrics_df.empty:
    print("No metrics.json files found under", LOCAL_OUTPUT_DIR)
else:
    display(metrics_df.sort_values(["model", "layer", "metrics_path", "split"]))
    summary_path = Path(DRIVE_OUTPUT_DIR) / f"dense_depth_{RUN_MODE}_metrics_summary.csv"
    metrics_df.to_csv(summary_path, index=False)
    print("Wrote summary:", summary_path)